In [1]:
import pandas as pd
import numpy as np 

In [2]:
df = pd.read_csv("Housing.csv")
df.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


In [3]:
df_encoded = pd.get_dummies(df, columns=["mainroad", "guestroom", "basement", "hotwaterheating", "airconditioning", "prefarea", "furnishingstatus"])
df_encoded.head()

,price,area,bedrooms,bathrooms,stories,parking,mainroad_no,mainroad_yes,guestroom_no,guestroom_yes,...,basement_yes,hotwaterheating_no,hotwaterheating_yes,airconditioning_no,airconditioning_yes,prefarea_no,prefarea_yes,furnishingstatus_furnished,furnishingstatus_semi-furnished,furnishingstatus_unfurnished
0,13300000,7420,4,2,3,2,False,True,True,False,...,False,True,False,False,True,False,True,True,False,False
1,12250000,8960,4,4,4,3,False,True,True,False,...,False,True,False,False,True,True,False,True,False,False
2,12250000,9960,3,2,2,2,False,True,True,False,...,True,True,False,True,False,False,True,False,True,False
3,12215000,7500,4,2,2,3,False,True,True,False,...,True,True,False,False,True,False,True,True,False,False
4,11410000,7420,4,1,2,2,False,True,False,True,...,True,True,False,False,True,True,False,True,False,False


In [4]:
np.sum(pd.isna(df_encoded) == True) # checking if any of the columns has NaN datapoints

C:\Users\mixas\anaconda3\Lib\site-packages\numpy\core\fromnumeric.py:86: FutureWarning: The behavior of DataFrame.sum with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return reduction(axis=axis, out=out, **passkwargs)


price                              0
area                               0
bedrooms                           0
bathrooms                          0
stories                            0
parking                            0
mainroad_no                        0
mainroad_yes                       0
guestroom_no                       0
guestroom_yes                      0
basement_no                        0
basement_yes                       0
hotwaterheating_no                 0
hotwaterheating_yes                0
airconditioning_no                 0
airconditioning_yes                0
prefarea_no                        0
prefarea_yes                       0
furnishingstatus_furnished         0
furnishingstatus_semi-furnished    0
furnishingstatus_unfurnished       0
dtype: int64

In [5]:
df_encoded = df_encoded.replace({True: 1, False: 0}).astype(float)
X = df_encoded.to_numpy()
X.dtype

C:\Users\mixas\AppData\Local\Temp\ipykernel_2504\3533865153.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_encoded = df_encoded.replace({True: 1, False: 0}).astype(float)


dtype('float64')

In [6]:
X = df_encoded.drop(["price"], axis = 1)
y = df_encoded["price"]
X.shape

(545, 20)

In [7]:
# Normalizing continuos data for better predictions
mean_area = X["area"].mean()
mean_price = y.mean()
std_area = X["area"].std()
std_price = y.std()
X["area"] = (X["area"] - mean_area) / std_area
y = (y - mean_price) / std_price


In [8]:
from sklearn.model_selection import train_test_split
# Split into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
# the model only takes numpy arrays so we need to conver dataframe to array
X_train = np.asarray(X_train.T)        # (features, m)
y_train = np.asarray(y_train).reshape(1, -1)  # (1, m) for BCE/MSE

In [10]:
from nn_constructor import NeuralNetwork, Layer, LinearAct, ReluAct, MeanRegressionError

# creating layers for the model
layer1 = Layer(in_features=20, out_features=16, activation=ReluAct())
layer2 = Layer(in_features=16, out_features=8, activation=ReluAct())
layer3 = Layer(in_features=8, out_features=4, activation=ReluAct() ) 
layer4 = Layer(in_features=4, out_features=1, activation=LinearAct() ) 

# Building network
layers = [layer1, layer2, layer3, layer4]

# Initialize loss
loss = MeanRegressionError(m=X_train.shape[1], y=y_train, lamda=0.01)

# Build NeuralNetwork object
nn = NeuralNetwork(alpha=0.1, data=X_train, loss=loss, layers=layers)

nn.fit()

epoch 1 - data_loss(MSE): 0.939353 | reg_loss: 0.000618 | total: 0.939970
epoch 2 - data_loss(MSE): 0.927706 | reg_loss: 0.000618 | total: 0.928324
epoch 3 - data_loss(MSE): 0.918209 | reg_loss: 0.000618 | total: 0.918827
epoch 4 - data_loss(MSE): 0.910451 | reg_loss: 0.000618 | total: 0.911068
epoch 5 - data_loss(MSE): 0.903926 | reg_loss: 0.000618 | total: 0.904543
epoch 6 - data_loss(MSE): 0.898423 | reg_loss: 0.000618 | total: 0.899041
epoch 7 - data_loss(MSE): 0.893717 | reg_loss: 0.000618 | total: 0.894335
epoch 8 - data_loss(MSE): 0.889747 | reg_loss: 0.000618 | total: 0.890365
epoch 9 - data_loss(MSE): 0.886245 | reg_loss: 0.000618 | total: 0.886863
epoch 10 - data_loss(MSE): 0.883184 | reg_loss: 0.000618 | total: 0.883802
epoch 11 - data_loss(MSE): 0.880305 | reg_loss: 0.000618 | total: 0.880923
epoch 12 - data_loss(MSE): 0.877524 | reg_loss: 0.000618 | total: 0.878142
epoch 13 - data_loss(MSE): 0.874550 | reg_loss: 0.000618 | total: 0.875168
epoch 14 - data_loss(MSE): 0.87150

In [11]:
X_test =np.asarray( X_test.T) 
prediction = nn.predict(X_test) # making predictions on X_test

In [12]:
y_test = np.asarray(y_test).reshape(1, -1)
nn.meansquarederror(prediction, y_test) # checking the accuracy of predictions on test data

0.5716184529706739